# Playing Doom with Sample Factory

Actor Critic models in Sample Factory are composed of three components:

* Encoder - Process input observations (images, vectors) and map them to a vector. This is the part of the model you will most likely want to customize.
* Core - Intergrate vectors from one or more encoders, can optionally include a single- or multi-layer LSTM/GRU in a memory-based agent.
* Decoder - Apply additional layers to the output of the model core before computing the policy and value outputs.

**ViZDoom** library ie an open-source python interface for the Doom Engine, enables the training of agents directly from sceen pixels. The library includes OpenAI Gymmnasium environmet wrappers and enables custom scenarios. It has access to the 3D vision depth buffer, audio buffer, automatic labeling of game obejects visible in frame, acces the list of objects and map geometry.


## <font size="18">**Health Gathering Supreme**</font>

### Install Dependencies

In [ ]:
!apt-get install build-essential zlib1g-dev libsdl2-dev libjpeg-dev \
nasm tar libbz2-dev libgtk2.0-dev cmake git libfluidsynth-dev libgme-dev \
libopenal-dev timidity libwildmidi-dev unzip ffmpeg

# Boost libraries
!apt-get install libboost-all-dev

# Lua binding dependencies
!apt-get install liblua5.1-dev

In [ ]:
!pip install sample-factory
!pip install vizdoom

In [ ]:
# verify installation
!python -m sample_factory.train.run_rl --help

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:
import gym
import numpy
import sample_factory
print("Gym:", gym.__version__)
print("NumPy:", numpy.__version__)

!pip show sample-factory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/ColabNotebooks/DeepRL/async_ppo_SampleFactory

Mounted at /content/drive
/content/drive/MyDrive/ColabNotebooks/DeepRL/async_ppo_SampleFactory


Logging in W&B or HF Hub can hang indefinitely in Colab if the notebook UI prompt doesn’t render correctly or if Colab’s I/O event loop gets stuck waiting for input use  CLI login

In [ ]:
!wandb login --relogin

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [ ]:
!huggingface-cli login

In [ ]:
import wandb

from huggingface_hub import notebook_login
from google.colab import userdata

# Retrieve API keys from Colab secrets
hf_token = userdata.get('HF_TOKEN')
wandb_key = userdata.get('WANDB_API_KEY')
#print("Loaded WANDB key:", bool(wandb_key))

notebook_login()
wandb.login(key=wandb_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

### Set Doom Environment within Sample Factory Framework

In [ ]:
import functools

from sample_factory.algo.utils.context import global_model_factory
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.env_utils import register_env
from sample_factory.train import run_rl

from sf_examples.vizdoom.doom.doom_model import make_vizdoom_encoder
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec


# Registers all the ViZDoom environments
def register_vizdoom_envs():
    for env_spec in DOOM_ENVS:
        make_env_func = functools.partial(make_doom_env_from_spec, env_spec)
        register_env(env_spec.name, make_env_func)


# Sample Factory allows the registration of a custom Neural Network architecture
# See https://github.com/alex-petrenko/sample-factory/blob/master/sf_examples/vizdoom/doom/doom_model.py for more details
def register_vizdoom_models():
  """ Register a custom neural network encoder for VizDoom environment with
      Sample Factory's models, so that the custom encoder is used when building
      the agent nn model. """
  global_model_factory().register_encoder_factory(make_vizdoom_encoder)


def register_vizdoom_components():
  """ Register environments and custom models with Sample Factory. """
  register_vizdoom_envs()
  register_vizdoom_models()


# parse the command line args and create a config
def parse_vizdoom_cfg(argv=None, evaluation=False):
  """Parsing command-line arguments and creats configuration object for training """
  parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
  # parameters specific to Doom envs
  add_doom_env_args(parser)
  # override Doom default values for algo parameters
  doom_override_defaults(parser)
  # second parsing pass yields the final configuration
  final_cfg = parse_full_cfg(parser, argv)
  return final_cfg

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

### Train

**Health Gathering Supreme**

The Agent learns how to survive without knowing what makes it survive. The map is a rectangle containing walls and with a green, acidic floor which hurts the player periodically. Initially there are some medkits spread uniformly over the map. A new medkit falls from the skies every now and then. Medkits heal some portions of player’s health - to survive, the agent needs to pick them up. The episode finishes after the player’s death or on timeout.

* Living_reward = 1
* 3 available buttons: turn left, turn right, move forward
* 1 available game variable: HEALTH
* death penalty = 100

More complex scenarios like include "doom_basic", "doom_two_colors_easy", "doom_dm", "doom_dwango5", "doom_my_way_home", "doom_deadly_corridor", "doom_defend_the_center", "doom_defend_the_line" at https://github.com/Farama-Foundation/ViZDoom/tree/master/scenarios

In [ ]:
register_vizdoom_components()

# train on HEalth Gathering Supreme
env = "doom_health_gathering_supreme"
# train for 5000000
cfg = parse_vizdoom_cfg(
    argv=[f"--env={env}",
          "--num_workers=8",
          "--num_envs_per_worker=4",
          "--experiment=doom_health_gathering_supreme_experiment",
          "--train_for_env_steps=5000000",
          "--with_wandb=True"]
)
# start training
status = run_rl(cfg)

[2025-10-17 19:39:28,531][03796] register_encoder_factory: <function make_vizdoom_encoder at 0x7d770bda45e0>
[2025-10-17 19:39:28,547][03796] Saved parameter configuration for experiment default_experiment not found!
[2025-10-17 19:39:28,548][03796] Starting experiment from scratch!
[2025-10-17 19:39:28,556][03796] Experiment dir /content/drive/MyDrive/ColabNotebooks/DeepRL/async_ppo_SampleFactory/train_dir/default_experiment already exists!
[2025-10-17 19:39:28,557][03796] Resuming existing experiment from /content/drive/MyDrive/ColabNotebooks/DeepRL/async_ppo_SampleFactory/train_dir/default_experiment...
[2025-10-17 19:39:28,558][03796] Weights and Biases integration enabled. Project: sample_factory, user: None, group: None, unique_id: default_experiment_20251017_193928_558125
[2025-10-17 19:39:28,559][03796] Initializing WandB...
wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


[2025-10-17 19:39:38,287][03796] Queried available GPUs: 0

/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
[2025-10-17 19:39:38,289][03796] Environment var CUDA_VISIBLE_DEVICES is 0

[2025-10-17 19:39:40,441][03796] Automatically setting recurrence to 32
[2025-10-17 19:39:40,443][03796] Starting experiment with the following configuration:
help=False
algo=APPO
env=doom_health_gathering_supreme
experiment=default_experiment
train_dir=/content/drive/MyDrive/ColabNotebooks/DeepRL/async_ppo_SampleFactory/train_dir
restart_behavior=resume
device=gpu
seed=None
num_policies=1
async_rl=True
serial_mode=False
batched_sampling=False
num_batches_to_accumulate=2
worker_num_splits=2
policy_workers_per_policy=1

### Resume Training

In [ ]:
register_vizdoom_components()

env = "doom_health_gathering_supreme"

cfg = parse_vizdoom_cfg(
    argv=[
        f"--env={env}",
        "--num_workers=8",
        "--num_envs_per_worker=4",
        "--train_for_env_steps=5000000",
        "--experiment=doom_health_gathering_supreme_experiment",
        "--train_dir=train_dir",
        "--with_wandb=True",
        "--restart_behavior=resume",        # to resume automatically
        "--load_checkpoint_kind=latest"     # tells Sample Factory to continue from the latest checkpoint
    ]
)

# start training
status = run_rl(cfg)

[2025-10-17 21:01:21,334][05354] Environment doom_basic already registered, overwriting...
[2025-10-17 21:01:21,336][05354] Environment doom_two_colors_easy already registered, overwriting...
[2025-10-17 21:01:21,338][05354] Environment doom_two_colors_hard already registered, overwriting...
[2025-10-17 21:01:21,340][05354] Environment doom_dm already registered, overwriting...
[2025-10-17 21:01:21,341][05354] Environment doom_dwango5 already registered, overwriting...
[2025-10-17 21:01:21,342][05354] Environment doom_my_way_home_flat_actions already registered, overwriting...
[2025-10-17 21:01:21,343][05354] Environment doom_defend_the_center_flat_actions already registered, overwriting...
[2025-10-17 21:01:21,344][05354] Environment doom_my_way_home already registered, overwriting...
[2025-10-17 21:01:21,346][05354] Environment doom_deadly_corridor already registered, overwriting...
[2025-10-17 21:01:21,346][05354] Environment doom_defend_the_center already registered, overwriting...

In [ ]:
!ls train_dir/doom_health_gathering_supreme_experiment

checkpoint_p0  config.json  replay.mp4	sf_log.txt


### Evaluation

Sample Factory internally calls checkpoint_dict = torch.load(latest_checkpoint, map_location=device), but in PyTorch 2.6+ defaults to weights_only=True, checkpoints created with PyTorch <2.6 contains only pickled metadata. Downgrade PpyTorch to < 2.6 or Monkey-patch torch.load back to old behavior torch.load(weights_only=False) with torch.load = lambda f, *a, **k: torch.serialization._legacy_load(open(f, 'rb'), map_location=k.get('map_location', None)) or explicitly allow NumPy's legacy scalar type
torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])


In [ ]:
!pip install torch==2.5.1

In [ ]:
!pip install sample-factory

In [ ]:
!pip install vizdoom

In [ ]:
import functools

from sample_factory.algo.utils.context import global_model_factory
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.env_utils import register_env
#from sample_factory.train import run_rl

from sf_examples.vizdoom.doom.doom_model import make_vizdoom_encoder
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec


# Registers all the ViZDoom environments
def register_vizdoom_envs():
    for env_spec in DOOM_ENVS:
        make_env_func = functools.partial(make_doom_env_from_spec, env_spec)
        register_env(env_spec.name, make_env_func)


def register_vizdoom_models():
  """ Register a custom neural network encoder for VizDoom environment with
      Sample Factory's models, so that the custom encoder is used when building
      the agent nn model. """
  global_model_factory().register_encoder_factory(make_vizdoom_encoder)


def register_vizdoom_components():
  """ Register environments and custom models with Sample Factory. """
  register_vizdoom_envs()
  register_vizdoom_models()


# parse the command line args and create a config
def parse_vizdoom_cfg(argv=None, evaluation=False):
  """Parsing command-line arguments and creats configuration object for training """
  parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
  # parameters specific to Doom envs
  add_doom_env_args(parser)
  # override Doom default values for algo parameters
  doom_override_defaults(parser)
  # second parsing pass yields the final configuration
  final_cfg = parse_full_cfg(parser, argv)
  return final_cfg


In [ ]:
!ls train_dir/doom_health_gathering_supreme_experiment

In [ ]:
from sample_factory.enjoy import enjoy

register_vizdoom_components()
env = "doom_health_gathering_supreme"

cfg = parse_vizdoom_cfg(
    argv=[f"--env={env}",
          "--num_workers=1",
          "--save_video",
          "--no_render",
          "--max_num_episodes=10",
          "--experiment=doom_health_gathering_supreme_experiment",
          "--train_dir=train_dir"],
    evaluation=True
)

# Sample Factory enjoy function runs evaluation episodes
status = enjoy(cfg)


In [ ]:
print(f"evaluation code: {status[0]}, \nmean episode reward: {status[1]:.2f}")

### Visualize

In [ ]:
from base64 import b64encode
from IPython.display import HTML

mp4 = open("train_dir/doom_health_gathering_supreme_experiment/replay.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(
    """
<video width=640 controls>
      <source src="%s" type="video/mp4">
</video>
"""
    % data_url
)


Output hidden; open in https://colab.research.google.com to view.

### Upload to HF Hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from sample_factory.enjoy import enjoy

hf_username = "elliemci"

cfg = parse_vizdoom_cfg(
    argv=[
        f"--env={env}",
        "--num_workers=1",
        "--save_video",
        "--no_render",
        "--max_num_episodes=10",
        "--max_num_frames=100000",
        "--push_to_hub",
        f"--hf_repository={hf_username}/doom_health_gathering_supreme",
    ],
    evaluation=True,
)
status = enjoy(cfg)
print(f"evaluation code: {status[0]}, \nmean episode reward: {status[1]:.2f}")

### Load from HF Hub

In [ ]:
#download the agent from the hub
!python -m sample_factory.huggingface.load_from_hub -r elliemci/doom_health_gathering_supreme -d ./train_dir

In [ ]:
# evaluate
env = "doom_health_gathering_supreme"
cfg = parse_vizdoom_cfg(
    argv=[
        f"--env={env}",
        "--num_workers=1",
        "--save_video",
        "--no_render",
        "--max_num_episodes=10",
        "--experiment=doom_health_gathering_supreme",
        "--train_dir=train_dir",
    ],
    evaluation=True,
)
status = enjoy(cfg)
print(f"evaluation code: {status[0]}, \nmean episode reward: {status[1]:.2f}")


## <font size="18">**Deadly Corridor**</font>

Scenario's purpose is to teach the agent to survive while navigating towards the green vest. The ao is a corridor with shooting mosters on both sides. The reward is proportional, negative or positive, to the change in the distance between the player and the vest.

### Setup for Sample Factory 2.1.1 + Gymnasium + VizDoom

In [ ]:
# --- System dependencies ---
!apt-get update -qq
!apt-get install -y build-essential zlib1g-dev libsdl2-dev libjpeg-dev \
    nasm tar libbz2-dev libgtk2.0-dev cmake git libfluidsynth-dev libgme-dev \
    libopenal-dev timidity libwildmidi-dev unzip ffmpeg \
    libboost-all-dev liblua5.1-dev

# --- Python packages ---
# Latest stable Sample Factory (Gymnasium-compatible)
!pip install -U "sample-factory[envs,vizdoom]" gymnasium wandb huggingface-hub

# ------- VizDoom ---
!pip install vizdoom

# visualization / debugging utilities
!pip install imageio[ffmpeg] moviepy


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/ColabNotebooks/DeepRL/async_ppo_SampleFactory

Mounted at /content/drive
/content/drive/MyDrive/ColabNotebooks/DeepRL/async_ppo_SampleFactory


In [ ]:
import gymnasium
import numpy
import sample_factory
print("Gym:", gym.__version__)
print("NumPy:", numpy.__version__)
#print("Sample Factory:", sample_factory.__version__)
!pip show sample-factory

In [ ]:
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS

print([e.name for e in DOOM_ENVS])

['doom_basic', 'doom_two_colors_easy', 'doom_two_colors_hard', 'doom_dm', 'doom_dwango5', 'doom_my_way_home_flat_actions', 'doom_defend_the_center_flat_actions', 'doom_my_way_home', 'doom_deadly_corridor', 'doom_defend_the_center', 'doom_defend_the_line', 'doom_health_gathering', 'doom_health_gathering_supreme', 'doom_battle', 'doom_battle2', 'doom_duel_bots', 'doom_deathmatch_bots', 'doom_duel', 'doom_deathmatch_full', 'doom_benchmark']


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# launch training with the new API for Sample Factory 2.x
!python -m sf_examples.vizdoom.train_vizdoom \
    --env=doom_deadly_corridor \
    --experiment=deadly_corridor_experiment \
    --train_dir=train_dir \
    --train_for_env_steps=5000000 \
    --with_wandb=True \
    --stats_avg=10 \
    --experiment_summaries_interval=60 \
    --flush_summaries_interval=10

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
[2025-10-19 00:34:07,414][06362] register_encoder_factory: <function make_vizdoom_encoder at 0x7eadfae43060>
[2025-10-19 00:34:07,428][06362] Loading existing experiment configuration from train_dir/deadly_corridor_experiment/config.json
[2025-10-19 00:34:07,430][06362] Overriding arg 'train_for_env_steps' with value 5000000 passed from command line
[2025-10-19 00:34:07,436][06362] Experiment dir train_dir/deadly_corridor_experiment already exists!
[2025-10-19 00:34:07,436][06362] Resuming existing experiment from train_dir/deadly_corridor_experiment...
[2025-10-19 00:34:07,436][06362] Weights and Biases integration enabled. Proj

In [ ]:
# stream the training log file
!tail -n 50 -f train_dir/doom_deadly_corridor/deadly_corridor_experiment/train.log


tail: cannot open 'train_dir/doom_deadly_corridor/deadly_corridor_experiment/train.log' for reading: No such file or directory
tail: no files remaining


In [ ]:
!ls train_dir/deadly_corridor_experiment/

checkpoint_p0  config.json  log.txt  sf_log.txt


In [ ]:
from sample_factory.enjoy import enjoy

register_vizdoom_components()
env = "doom_deadly_corridor"

cfg = parse_vizdoom_cfg(
    argv=[f"--env={env}",
          "--num_workers=1",
          "--max_num_episodes=10",
          "--save_video",
          "--no_render",
          "--experiment=deadly_corridor_experiment",
          "--train_dir=train_dir"],
    evaluation=True
)

status = enjoy(cfg)  # cfg must have evaluation=True

print("evaluation code:", status[0], "mean reward:", status[1])

Note: PyTorch 2.6 changed torch.load() so that by default it only allows "weights-only" loading which prevents untristed objects from being unpickled. In old SF checkpoint contains objects like numpy.core.multiarray.scalar that are no longer allowed by default, and Learnier.load_checkpoint() keeps failing and returning None. Can fix it be lading weights_only=False, allow list of unsafe globals with

```python
import torch
import numpy

with torch.serialization.add_safe_globals([numpy.core.multiarray.scalar]):
    checkpoint_dict = torch.load("path/to/checkpoint.pth", map_location="cpu", weights_only=True)
```

or downgrade PyTorch to 2.5.1<br>
!pip install torch==2.5.1+cu118 --extra-index-url https://download.pytorch.org/whl/cu118


In [ ]:
import torch
from sample_factory.algo.learning.learner import Learner

# Save original method
_original_load_checkpoint = Learner.load_checkpoint

# Patch the method to force weights_only=False
def patched_load_checkpoint(checkpoints, device):
    latest_checkpoint = checkpoints[-1]  # take the latest checkpoint
    print(f"Loading checkpoint with weights_only=False: {latest_checkpoint}")
    # Use weights_only=False to allow loading old SF checkpoints
    checkpoint_dict = torch.load(latest_checkpoint, map_location=device, weights_only=False)
    return checkpoint_dict

# Apply the patch
Learner.load_checkpoint = staticmethod(patched_load_checkpoint)

# Now you can evaluate as usual
from sample_factory.enjoy import enjoy
status = enjoy(cfg)  # cfg must have evaluation=True

print(f"evaluation code: {status[0]}, mean reward: {status[1]:.2f}")


[2025-10-19 03:06:52,359][02420] Loading existing experiment configuration from train_dir/deadly_corridor_experiment/config.json
[2025-10-19 03:06:52,360][02420] Adding new argument 'no_render'=True that is not in the saved config file!
[2025-10-19 03:06:52,362][02420] Adding new argument 'save_video'=True that is not in the saved config file!
[2025-10-19 03:06:52,363][02420] Adding new argument 'video_frames'=1000000000.0 that is not in the saved config file!
[2025-10-19 03:06:52,364][02420] Adding new argument 'video_name'=None that is not in the saved config file!
[2025-10-19 03:06:52,365][02420] Adding new argument 'max_num_frames'=1000000000.0 that is not in the saved config file!
[2025-10-19 03:06:52,366][02420] Adding new argument 'max_num_episodes'=10 that is not in the saved config file!
[2025-10-19 03:06:52,367][02420] Adding new argument 'push_to_hub'=False that is not in the saved config file!
[2025-10-19 03:06:52,368][02420] Adding new argument 'hf_repository'=None that is

Loading checkpoint with weights_only=False: train_dir/deadly_corridor_experiment/checkpoint_p0/checkpoint_000001030_4218880.pth


[2025-10-19 03:06:54,456][02420] Num frames 100...
[2025-10-19 03:06:54,598][02420] Num frames 200...
[2025-10-19 03:06:54,684][02420] Avg episode rewards: #0: 22.784, true rewards: #0: 22.784
[2025-10-19 03:06:54,685][02420] Avg episode reward: 22.784, avg true_objective: 22.784
[2025-10-19 03:06:54,887][02420] Num frames 300...
[2025-10-19 03:06:55,055][02420] Avg episode rewards: #0: 14.828, true rewards: #0: 14.828
[2025-10-19 03:06:55,057][02420] Avg episode reward: 14.828, avg true_objective: 14.828
[2025-10-19 03:06:55,156][02420] Num frames 400...
[2025-10-19 03:06:55,377][02420] Avg episode rewards: #0: 12.194, true rewards: #0: 12.194
[2025-10-19 03:06:55,378][02420] Avg episode reward: 12.194, avg true_objective: 12.194
[2025-10-19 03:06:55,414][02420] Num frames 500...
[2025-10-19 03:06:55,616][02420] Num frames 600...
[2025-10-19 03:06:55,846][02420] Avg episode rewards: #0: 14.838, true rewards: #0: 14.838
[2025-10-19 03:06:55,847][02420] Avg episode reward: 14.838, avg t

evaluation code: 0, mean reward: 10.73


In [ ]:
from base64 import b64encode
from IPython.display import HTML

# visualize
mp4 = open("train_dir/deadly_corridor_experiment/replay.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(
    """
<video width=640 controls>
      <source src="%s" type="video/mp4">
</video>
"""
    % data_url
)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sample_factory.enjoy import enjoy

hf_username = "elliemci"

cfg = parse_vizdoom_cfg(
    argv=[
        f"--env={env}",
        "--num_workers=1",
        "--save_video",
        "--no_render",
        "--max_num_episodes=10",
        "--max_num_frames=100000",
        "--push_to_hub",
        f"--hf_repository={hf_username}/deadly_corridor_experiment",
        "--experiment=deadly_corridor_experiment" # required for the creation of local folder structure and checkpoint paths
    ],
    evaluation=True,
)
status = enjoy(cfg)
print(f"evaluation code: {status[0]}, \nmean episode reward: {status[1]:.2f}")

[2025-10-19 03:25:20,903][02420] Loading existing experiment configuration from /content/drive/MyDrive/ColabNotebooks/DeepRL/async_ppo_SampleFactory/train_dir/deadly_corridor_experiment/config.json
[2025-10-19 03:25:20,905][02420] Adding new argument 'no_render'=True that is not in the saved config file!
[2025-10-19 03:25:20,905][02420] Adding new argument 'save_video'=True that is not in the saved config file!
[2025-10-19 03:25:20,907][02420] Adding new argument 'video_frames'=1000000000.0 that is not in the saved config file!
[2025-10-19 03:25:20,908][02420] Adding new argument 'video_name'=None that is not in the saved config file!
[2025-10-19 03:25:20,909][02420] Adding new argument 'max_num_frames'=100000 that is not in the saved config file!
[2025-10-19 03:25:20,910][02420] Adding new argument 'max_num_episodes'=10 that is not in the saved config file!
[2025-10-19 03:25:20,911][02420] Adding new argument 'push_to_hub'=True that is not in the saved config file!
[2025-10-19 03:25:2

Loading checkpoint with weights_only=False: train_dir/deadly_corridor_experiment/checkpoint_p0/checkpoint_000001030_4218880.pth


[2025-10-19 03:25:21,478][02420] Num frames 100...
[2025-10-19 03:25:21,605][02420] Avg episode rewards: #0: 7.455, true rewards: #0: 7.455
[2025-10-19 03:25:21,606][02420] Avg episode reward: 7.455, avg true_objective: 7.455
[2025-10-19 03:25:21,677][02420] Num frames 200...
[2025-10-19 03:25:21,869][02420] Avg episode rewards: #0: 7.430, true rewards: #0: 7.430
[2025-10-19 03:25:21,869][02420] Avg episode reward: 7.430, avg true_objective: 7.430
[2025-10-19 03:25:21,874][02420] Num frames 300...
[2025-10-19 03:25:21,984][02420] Avg episode rewards: #0: 5.145, true rewards: #0: 5.145
[2025-10-19 03:25:21,985][02420] Avg episode reward: 5.145, avg true_objective: 5.145
[2025-10-19 03:25:22,072][02420] Num frames 400...
[2025-10-19 03:25:22,224][02420] Num frames 500...
[2025-10-19 03:25:22,291][02420] Avg episode rewards: #0: 5.824, true rewards: #0: 5.824
[2025-10-19 03:25:22,291][02420] Avg episode reward: 5.824, avg true_objective: 5.824
[2025-10-19 03:25:22,427][02420] Num frames 6

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...s.1760762395.7f51b3d89358:   1%|1         | 5.33kB /  358kB            

... (5 outputs left out)

  ...int_000001030_4218880.pth:   2%|1         |  562kB / 35.0MB            

  ...4288512_reward_15.226.pth:   2%|1         |  559kB / 35.0MB            

[2025-10-19 03:25:48,066][02420] The model has been pushed to https://huggingface.co/elliemci/deadly_corridor_experiment


evaluation code: 0, 
mean episode reward: 8.92
